# Forecasting Weekly Demand
### Kestrel Garden & Outdoor: how many barbecues should we order?

---

Kestrel Garden & Outdoor sells barbecues, plants, tools and pots. Before the coming summer, the buying team has to decide how many barbecues to order. Three questions sit behind that one order:

1. How many units will we sell each week?
2. How wrong might we be?
3. What does being wrong cost?

In this notebook you turn the shop's sales records into a weekly demand series, forecast it, score the forecast on a year the model has not seen, and use it to decide how much stock to order. Then you repeat the work on a second category and see which drivers of demand matter where.

**The data is synthetic.** `sales_lines.csv` and `weekly_calendar.csv` were generated for this module. Kestrel is fictional, and so are its prices, promotions and weather. The patterns behave like real retail demand. The numbers do not belong to a real shop.

---

### The AI rule

Copilot is on. Use it to draft code, then check what it gives you against the numbers printed in this notebook and the numbers JMP gives you. A forecast you cannot check is a guess with a tidy chart.

### Before you start

- Open this notebook in your Codespace.
- The data files are in `../data/`.
- Run cells in order with **Shift+Enter**.
- Files written for JMP go to an `exports/` folder.

### How each part works

Every part follows the same five steps:

1. **Load and look**: see the data before you model it.
2. **Run the model**: fit it and score it.
3. **Read the output in plain English**: say what the numbers mean.
4. **Check against JMP**: reproduce a key number in a second tool.
5. **Say it to a manager**: two sentences, no jargon.

## Setup: libraries and fixed settings

Everything below uses the same settings, so every number you see can be traced back to this one cell. The split is by time: the model learns from the first 105 weeks and is tested on the last 52. Nothing is ever shuffled, because the test year is the future.

In [ ]:
# Libraries: pandas for tables, sqlite3 for SQL, statsmodels for regression, matplotlib for charts
import os
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.model_selection import ShuffleSplit   # used once, to show why a shuffled split flatters a forecast

os.makedirs("exports", exist_ok=True)               # files written for JMP go here

# Fixed settings, used in every part
TRAIN_WEEKS = 105      # first 105 weeks: 2 Jan 2023 to 30 Dec 2024
TEST_WEEKS = 52        # last 52 weeks: 6 Jan 2025 to 29 Dec 2025
SHORT_COST = 12        # invented cost, in pounds, of each unit we are short
EXCESS_COST = 4        # invented cost, in pounds, of each unit left unsold
CRITICAL_RATIO = SHORT_COST / (SHORT_COST + EXCESS_COST)   # 0.75: order enough to cover demand in 3 weeks out of 4
Z_075 = 0.6745         # how many standard deviations above the forecast the 0.75 point sits

print("Synthetic data: Kestrel Garden & Outdoor is fictional.")
print(f"Train on the first {TRAIN_WEEKS} weeks, test on the last {TEST_WEEKS}. Critical ratio = {CRITICAL_RATIO:.2f}.")

---
## Part 1: Demo: baselines, trend and season, promotions and weather (barbecue)

**The business question:** how many barbecues will we sell each week, and how well can we forecast them?

### Step 1: Load and look

The sales file has one row per sale line. The buying team needs one row per week, so we group the lines by the Monday that starts their week. The query below does that. **Read it aloud before you run it**: what does each line ask SQLite to do?

In [ ]:
# Load both tables into an in-memory SQLite database
sales = pd.read_csv("../data/sales_lines.csv")
calendar = pd.read_csv("../data/weekly_calendar.csv")
con = sqlite3.connect(":memory:")
sales.to_sql("sales", con, index=False)
calendar.to_sql("calendar", con, index=False)

print(f"{len(sales):,} sale lines, {len(calendar)} weeks in the calendar, {sales['Quantity'].sum():,} units in total")

In [ ]:
# One row per calendar week. LEFT JOIN keeps every week, even one with no sales lines.
# date(SaleDate, 'weekday 0', '-6 days') turns any sale date into the Monday that starts its week.
WEEKLY_SQL = """
SELECT c.WeekStart, c.AvgTempC, c.RainDays, c.SchoolHoliday, c.PromoActive, c.PromoDiscountPct,
       COALESCE(SUM(s.Quantity), 0) AS Units
FROM calendar AS c
LEFT JOIN sales AS s
       ON date(s.SaleDate, 'weekday 0', '-6 days') = c.WeekStart
      AND s.Category = ?
GROUP BY c.WeekStart
ORDER BY c.WeekStart
"""

def build_weekly(category):
    """Weekly units for one category, plus the columns the models use."""
    d = pd.read_sql(WEEKLY_SQL, con, params=[category])
    d["date"] = pd.to_datetime(d["WeekStart"])
    d["t"] = np.arange(len(d))                                   # trend: week number from 0
    day = d["date"].dt.dayofyear
    for k in (1, 2):                                              # season: first and second harmonics
        d[f"s{k}"] = np.sin(2 * np.pi * k * day / 365.25)
        d[f"c{k}"] = np.cos(2 * np.pi * k * day / 365.25)
    d["promo_pts"] = d["PromoActive"] * d["PromoDiscountPct"]    # promotion size in percentage points
    d["Split"] = np.where(d["t"] < TRAIN_WEEKS, "train", "test")
    d["Validation"] = (d["t"] >= TRAIN_WEEKS).astype(int)        # 0 = train, 1 = test (for JMP)
    return d

barbecue = build_weekly("Barbecue and Dining")
print(barbecue.shape)
print(barbecue[["WeekStart", "Units", "AvgTempC", "RainDays", "SchoolHoliday", "promo_pts", "Split"]].head().to_string(index=False))

In [ ]:
# Check the series before trusting it: 157 weeks, and the units must add up to the sales file
barbecue_lines = sales[sales["Category"] == "Barbecue and Dining"]["Quantity"].sum()
print("Weeks:", len(barbecue), "| units in the weekly series:", int(barbecue["Units"].sum()), "| units in the sales file:", int(barbecue_lines))
print("Smallest week:", int(barbecue["Units"].min()), "| largest week:", int(barbecue["Units"].max()), "| mean:", round(barbecue["Units"].mean(), 1))

In [ ]:
# Look at the whole series, with promotion weeks marked and the test year shaded
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(barbecue["date"], barbecue["Units"], color="#1A4C3B", linewidth=1.5)
promo_weeks = barbecue[barbecue["PromoActive"] == 1]
ax.scatter(promo_weeks["date"], promo_weeks["Units"], color="#E07B39", zorder=3, label="promotion week")
ax.axvspan(barbecue.loc[TRAIN_WEEKS, "date"], barbecue["date"].iloc[-1], color="grey", alpha=0.15, label="test year")
ax.set_ylabel("Units sold in the week")
ax.set_title("Barbecue and Dining: weekly units")
ax.legend()
plt.show()

In [ ]:
# Promotion weeks against the rest. The gap is real, but it is not the promotion effect on its own.
by_promo = barbecue.groupby("PromoActive")["Units"].agg(["count", "mean"]).round(1)
by_promo.index = ["no promotion", "promotion"]
print(by_promo.to_string())
print("\nPromotions are not spread evenly across the year, so this gap mixes the promotion with the season it ran in.")

In [ ]:
# The split: the first 105 weeks train the model, the last 52 are the future it has to forecast
train = barbecue[barbecue["Split"] == "train"]
test = barbecue[barbecue["Split"] == "test"]
print("Train:", train["WeekStart"].iloc[0], "to", train["WeekStart"].iloc[-1], f"({len(train)} weeks)")
print("Test: ", test["WeekStart"].iloc[0], "to", test["WeekStart"].iloc[-1], f"({len(test)} weeks)")

### Your turn: look before you model

Change `"Barbecue and Dining"` in the `build_weekly` call to `"Pots and Planters"` and re-run the plot cell. Does the shape of the year look the same? Change it back before you go on.

### Step 2: Run the model

We score seven ways of forecasting the 52 test weeks. Four are simple and need no model. Three are regressions that add information step by step. One function, `table_row`, scores every method the same way:

- **MAE**: the average miss, in units.
- **MAPE**: the average miss as a percentage of what actually sold.
- **Bias**: the average of forecast minus actual. Positive means we over-forecast on average.

In [ ]:
# One scoring function for every method, so every row of the table is comparable
def score(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    miss = np.asarray(forecast, dtype=float) - actual
    return {"MAE": np.mean(np.abs(miss)),
            "MAPE %": np.mean(np.abs(miss) / actual) * 100,
            "Bias": np.mean(miss)}

def table_row(name, actual, forecast):
    return {"Method": name, **score(actual, forecast)}

# The drivers a regression can use, and the model definitions built from them
DRIVERS = {
    "trend": ["t"],
    "season": ["s1", "c1", "s2", "c2"],
    "promotions": ["promo_pts"],
    "school holidays": ["SchoolHoliday"],
    "rain days": ["RainDays"],
    "temperature": ["AvgTempC"],
}

def columns_for(names):
    return [col for name in names for col in DRIVERS[name]]

models = {
    "Regression: trend and season":
        columns_for(["trend", "season"]),
    "Regression: trend, season, promotions, school holidays, rain days":
        columns_for(["trend", "season", "promotions", "school holidays", "rain days"]),
    "Regression: trend, season, promotions, school holidays, rain days, temperature":
        columns_for(["trend", "season", "promotions", "school holidays", "rain days", "temperature"]),
}

def fit_regression(d, columns):
    """Fit on the training weeks only, then forecast the test weeks from their own driver values."""
    tr, te = d[d["t"] < TRAIN_WEEKS], d[d["t"] >= TRAIN_WEEKS]
    model = sm.OLS(tr["Units"], sm.add_constant(tr[columns])).fit()
    forecast = model.predict(sm.add_constant(te[columns], has_constant="add"))
    return model, forecast

In [ ]:
def compare_methods(d):
    """Score the four simple methods and the three regressions on the test year."""
    y = d["Units"].to_numpy()
    test_weeks = np.arange(TRAIN_WEEKS, len(d))
    actual = y[test_weeks]
    rows = [
        table_row("Last week's value (naive)", actual, y[test_weeks - 1]),
        table_row("Same week last year", actual, y[test_weeks - 52]),
        table_row("Mean of last four weeks", actual, [y[i - 4:i].mean() for i in test_weeks]),
    ]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        hw = ExponentialSmoothing(y[:TRAIN_WEEKS], trend="add", seasonal="add", seasonal_periods=52).fit()
    rows.append(table_row("Holt-Winters (trend and season, 52-week cycle)", actual, hw.forecast(TEST_WEEKS)))
    for name, columns in models.items():
        rows.append(table_row(name, actual, fit_regression(d, columns)[1]))
    return pd.DataFrame(rows)

barbecue_table = compare_methods(barbecue)
print(barbecue_table.round(1).to_string(index=False))

In [ ]:
# Which method has the lowest average percentage miss on the test year?
best_name = barbecue_table.loc[barbecue_table["MAPE %"].idxmin(), "Method"]
print("Lowest MAPE:", best_name)

### Your turn: change a setting and re-run

The "mean of last four weeks" baseline averages the previous 4 weeks. In `compare_methods`, change `y[i - 4:i]` to `y[i - 8:i]` to average 8 weeks instead, run the cell that defines the function, then run `compare_methods(barbecue)` again. Did a longer memory help or hurt? Put the 4 back afterwards.

**Copilot:** ask it to *"add a baseline to compare_methods that forecasts each test week as the average of the same week in the two earlier years"*. Check its answer: does it only use weeks before the one it is forecasting?

### Step 3: Read the output in plain English

Three things to read: the table, the picture, and what the regression says about each driver.

In [ ]:
# The table in words, built from the numbers above so the sentences always match the table
simple = barbecue_table.iloc[:4]
best_simple = simple.loc[simple["MAPE %"].idxmin()]
regs = barbecue_table.iloc[4:]
worst_reg, best_reg = regs.iloc[0], regs.loc[regs["MAPE %"].idxmin()]
print(f"The best simple method is '{best_simple['Method']}': it misses by {best_simple['MAE']:.1f} units a week on average, {best_simple['MAPE %']:.1f}% of what sold.")
print(f"A regression with only trend and season misses by {worst_reg['MAE']:.1f} units ({worst_reg['MAPE %']:.1f}%).")
print(f"The best regression, '{best_reg['Method']}', misses by {best_reg['MAE']:.1f} units ({best_reg['MAPE %']:.1f}%).")
print(f"Its bias is {best_reg['Bias']:+.1f} units: on average it {'over' if best_reg['Bias'] > 0 else 'under'}-forecasts a week by that much.")

In [ ]:
# The best regression against what actually sold in the test year
best_columns = models[best_name]
best_model, best_forecast = fit_regression(barbecue, best_columns)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test["date"], test["Units"], color="#1A4C3B", linewidth=1.8, label="actual")
ax.plot(test["date"], best_forecast, color="#E07B39", linewidth=1.8, linestyle="--", label="forecast")
ax.set_ylabel("Units sold in the week")
ax.set_title("Test year: forecast against actual")
ax.legend()
plt.show()

In [ ]:
# What the regression says about each driver. p-value: how surprising the coefficient would be if the driver did nothing.
coef = pd.DataFrame({"coefficient": best_model.params, "p-value": best_model.pvalues})
print(coef.round(4).to_string())
print()
promo = best_model.params["promo_pts"]
print(f"Each percentage point of discount adds about {promo:.1f} units in a promotion week, so a 15% promotion adds about {15 * promo:.0f} units.")
if "AvgTempC" in best_model.params:
    print(f"Each extra degree of average weekly temperature goes with about {best_model.params['AvgTempC']:.1f} more units.")
print(f"Durbin-Watson on the training residuals: {durbin_watson(best_model.resid):.2f}")

**Durbin-Watson** checks for *autocorrelation*: whether a week's miss tells you something about the following week's miss. A value near 2 says it does not. Values near 0 or 4 say the misses are running in streaks, and the model is missing something that changes slowly.

### Your turn: read a coefficient

Use the coefficient table to answer: roughly how many extra units would a 20% promotion bring in a week? Then check your answer by multiplying out `promo_pts` yourself.

**Stretch:** the trend coefficient is small and its p-value is not tiny. What would you tell a buyer who asked, "are barbecues growing?"

### Step 4: Check against JMP

JMP is the second pair of eyes. The cell below writes the weekly table to `exports/` with two extra columns: `Validation` (0 for the 105 training weeks, 1 for the 52 test weeks) and `Split` (the same thing in words). Follow the Activity Workbook on forecasting in JMP, fit the same model, and compare.

In [ ]:
# Write the weekly table for JMP: a numeric Validation column (0 train, 1 test) and a Split label
export_columns = ["WeekStart", "Split", "Validation", "t", "s1", "c1", "s2", "c2",
                  "promo_pts", "SchoolHoliday", "RainDays", "AvgTempC", "Units"]
barbecue[export_columns].to_csv("exports/barbecue_weekly_for_jmp.csv", index=False)
print("Written: exports/barbecue_weekly_for_jmp.csv", barbecue[export_columns].shape)

In [ ]:
# Python numbers for the JMP check: training R-squared and test-year MAE for each regression
jmp_rows = []
for name, columns in models.items():
    model, forecast = fit_regression(barbecue, columns)
    jmp_rows.append({"Model": name, "Training R-squared": model.rsquared,
                     "Test MAE": score(test["Units"], forecast)["MAE"]})
print(pd.DataFrame(jmp_rows).round(3).to_string(index=False))

JMP's training R-squared should match to three decimals. Test MAE can differ a little because JMP handles the validation column in its own way. If the difference is more than a few units, check the column list first.

### Step 5: Say it to a manager

Two sentences, no jargon. The cell below drafts them from the numbers. Rewrite them in your own words in the cell underneath, as if the buying director had stopped you in the corridor.

In [ ]:
# A draft built from the numbers. Your own words go in the next cell.
print(f"We can forecast weekly barbecue demand to within about {best_reg['MAE']:.0f} units a week, which is {best_reg['MAPE %']:.1f}% of what we sell.")
print(f"A simple 'same as last year' guess misses by {barbecue_table.loc[barbecue_table['Method'] == 'Same week last year', 'MAPE %'].iloc[0]:.1f}%, so the model earns its keep.")

*Your two sentences here*

### Commentary: Part 1

1. Explain MAE, MAPE and bias to someone who sells barbecues, in one sentence each.
2. The regression beats "same week last year". What does a regression know that last year's number does not?
3. Barbecue weeks with a promotion average far more units than weeks without. Why is that gap not the effect of the promotion?

*Your commentary here*

**Stretch:** ask Copilot to plot the forecast error (actual minus forecast) for each test week as a bar chart. Which weeks are furthest out, and what was happening in them?

In [ ]:
# Stretch

---
## Part 2: Demo: from forecast to stock order (barbecue)

**The business question:** the buying team will order from this forecast. Is it safe to, how wrong might it be, and how much should we order?

### Step 1: Load and look

Before any stock is ordered, ask one question about every driver in the model: **on the day the order is placed, would you know its value for the weeks you are ordering for?**

Fill in the dictionary with `True` or `False` for each driver, then run the cell. It rebuilds the model from the drivers you marked `True` and scores it. If you leave a `None`, the cell asks you to finish and the model stays as it was.

In [ ]:
# The model we are about to order from starts as the one with the lowest MAPE in Part 1
final_columns = list(models[best_name])
print("Model in use:", ", ".join(final_columns))

In [ ]:
# Your answers: True if you would know this driver for each week when the order is placed, False if you would not
known_when_ordering = {
    "trend": None,
    "season": None,
    "promotions": None,
    "school holidays": None,
    "rain days": None,
    "temperature": None,
}

if None in known_when_ordering.values():
    print("Fill in every True or False above, then run this cell again. The model in use has not changed.")
else:
    allowed = [name for name, known in known_when_ordering.items() if known]
    final_columns = columns_for(allowed)
    rebuilt = score(test["Units"], fit_regression(barbecue, final_columns)[1])
    print("Model rebuilt from:", ", ".join(allowed))
    print(f"Test-year MAPE: {rebuilt['MAPE %']:.1f}%   MAE: {rebuilt['MAE']:.1f} units")

**Rain days is a judgement call.** A buyer only has a long-range guess for it, not the real count. Decide what you would do, and say why in the commentary. If you mark it `False`, your numbers will differ a little from a classmate who marks it `True`. Both are defensible if you can explain the choice.

After any change to `final_columns`, **run this cell and every cell below it** so that all later numbers use the same model.

### Step 2: Run the model

From here the forecast is a *number of units*, and an *interval* is a range of units around it. We measure how widely the model has missed on the training weeks, then ask three questions: how often does the actual fall inside an interval, what does each ordering policy cost, and which one is cheapest?

In [ ]:
# Forecast the test year with the model in use, and measure the typical miss on the training weeks
model, forecast = fit_regression(barbecue, final_columns)
actual = test["Units"].to_numpy()
residual_sd = np.sqrt(model.mse_resid)       # standard deviation of the training misses, in units
print("Model in use:", ", ".join(final_columns))
print(f"Residual standard deviation on the training weeks: {residual_sd:.1f} units")

In [ ]:
# Do the intervals cover as often as they claim? Count the test weeks that land inside each one.
for level in (0.80, 0.90):
    z = norm.ppf(0.5 + level / 2)
    inside = ((actual >= forecast - z * residual_sd) & (actual <= forecast + z * residual_sd)).sum()
    print(f"A nominal {level:.0%} interval covered {inside} of {TEST_WEEKS} test weeks ({inside / TEST_WEEKS:.1%}).")

In [ ]:
# Three ways to turn a forecast into an order. Cost = 12 per unit short + 4 per unit left over.
def stock_cost(order, actual):
    short = np.maximum(actual - order, 0)
    excess = np.maximum(order - actual, 0)
    return {"Units short": short.sum(), "Units left over": excess.sum(),
            "Cost (£)": SHORT_COST * short.sum() + EXCESS_COST * excess.sum(),
            "Weeks demand covered": (order >= actual).mean() * 100}

uplift = Z_075 * residual_sd                  # the extra units the 0.75 rule adds to the forecast
last_year = barbecue["Units"].to_numpy()[np.arange(TRAIN_WEEKS, len(barbecue)) - 52]
policies = {
    "Order the forecast": forecast.to_numpy(),
    f"Order forecast + {uplift:.1f} (0.75 point)": forecast.to_numpy() + uplift,
    "Order the same week last year": last_year.astype(float),
}
policy_table = pd.DataFrame({name: stock_cost(order, actual) for name, order in policies.items()}).T
print(policy_table.round(1).to_string())
cheapest = policy_table["Cost (£)"].idxmin()
print("\nCheapest policy:", cheapest)

### Your turn: move the order point

The 0.75 rule comes from the costs: 12 / (12 + 4) = 0.75. Change `SHORT_COST` to 6 in the setup cell, re-run the setup cell and then this part from `stock_cost` down. Does the cheapest uplift get bigger or smaller? Put the cost back to 12 afterwards.

**Copilot:** ask it to *"plot total cost against the number of extra units added to the forecast, from 0 to 100"*. Check that the lowest point sits close to the uplift above. If it does not, find out why before you trust the plot.

In [ ]:
# Copilot's cost curve goes here

### Step 3: Read the output in plain English

The weeks the model misses most, and one trap in how the test year is chosen.

In [ ]:
# The five weeks with the biggest misses: what was happening in them?
worst = test.assign(Forecast=forecast, Miss=forecast - actual)
worst["Absolute miss"] = worst["Miss"].abs()
cols = ["WeekStart", "Units", "Forecast", "Miss", "AvgTempC", "PromoActive", "SchoolHoliday"]
print(worst.sort_values("Absolute miss", ascending=False)[cols].head(5).round(1).to_string(index=False))

In [ ]:
# Reading the cost table and the intervals in words, from the numbers above
point, quantile = policy_table.iloc[0], policy_table.iloc[1]
print(f"Ordering the forecast costs £{point['Cost (£)']:,.0f}: {point['Units short']:,.0f} units short and {point['Units left over']:,.0f} left over, with demand covered in {point['Weeks demand covered']:.1f}% of weeks.")
print(f"Ordering {uplift:.1f} units above the forecast costs £{quantile['Cost (£)']:,.0f}: {quantile['Units short']:,.0f} short, {quantile['Units left over']:,.0f} left over, covered in {quantile['Weeks demand covered']:.1f}% of weeks.")
print(f"Last year's number costs £{policy_table.iloc[2]['Cost (£)']:,.0f}.")
print("Being short costs three times as much per unit as being long, so the cheapest policy deliberately orders more than the forecast.")

In [ ]:
# A trap, shown once: pick the test weeks at random instead of taking the last 52.
# Neighbouring weeks end up on both sides of the split, so the model has already seen the weeks next to each test week.
actual_all = barbecue["Units"].to_numpy()
shuffled_mape = []
for seed in range(5):
    train_idx, test_idx = next(ShuffleSplit(n_splits=1, test_size=TEST_WEEKS, random_state=seed).split(barbecue))
    fit = sm.OLS(barbecue["Units"].iloc[train_idx], sm.add_constant(barbecue[final_columns].iloc[train_idx])).fit()
    pred = fit.predict(sm.add_constant(barbecue[final_columns].iloc[test_idx], has_constant="add"))
    shuffled_mape.append(score(actual_all[test_idx], pred)["MAPE %"])

real_mape = score(actual, forecast)["MAPE %"]
print("MAPE on five shuffled 52-week test sets:", [round(float(m), 1) for m in shuffled_mape])
print(f"Range {min(shuffled_mape):.1f}% to {max(shuffled_mape):.1f}%, mean {np.mean(shuffled_mape):.1f}%, against {real_mape:.1f}% on the real test year.")
print("Compare the two. In a shuffled split the weeks either side of each test week are in the training data, so the model has already seen the neighbours of every week it is scored on.")

### Your turn: how much did the interval miss?

An interval claimed to cover a share of weeks and covered a different share. In your own words, write a fair one-line statement for the buying team, using the counts printed in Step 2 ("about N weeks in 10, not M in 10").

### Step 4: Check against JMP

JMP's fit report gives a *Root Mean Square Error* for the training weeks. It should match the residual standard deviation printed in Step 2, because both divide by the number of weeks minus the number of terms in the model.

In [ ]:
# The numbers to find in JMP: the model's training R-squared and its root mean square error
print(f"Model in use: {', '.join(final_columns)}")
print(f"Training R-squared: {model.rsquared:.3f}")
print(f"Root mean square error (the residual standard deviation): {residual_sd:.1f} units")

### Step 5: Say it to a manager

Two sentences for the buying director: what we expect, and what to order.

In [ ]:
# A draft built from the numbers. Your own words go in the next cell.
print(f"We expect to be out by about {score(actual, forecast)['MAE']:.0f} units a week, and the 80% interval we would quote covered only {((actual >= forecast - norm.ppf(0.9) * residual_sd) & (actual <= forecast + norm.ppf(0.9) * residual_sd)).mean():.0%} of the test weeks.")
print(f"Because running short costs three times as much as holding extra, I would order about {uplift:.0f} units a week above the forecast, which cost £{quantile['Cost (£)']:,.0f} over the test year against £{point['Cost (£)']:,.0f} for ordering the forecast.")

*Your two sentences here*

### Commentary: Part 2

1. For each driver in the model, say when you would know its value, and what you decided to do about it.
2. Why is the cheapest ordering policy not the one with the most accurate forecast?
3. One test year is one year. Name one thing that could make next year's result different.

*Your commentary here*

**Stretch:** replace the temperature with the *average temperature for that week of the year in the two training years* and score the model. Is a stand-in for a driver you cannot know better or worse than ignoring the driver?

In [ ]:
# Stretch

---
## Part 3: Exercise: the Tools category

**The business question:** do the same drivers matter for tools as for barbecues? The functions you wrote in Parts 1 and 2 do the work. You steer them.

### Step 1: Load and look

Build the weekly series for the Tools category with the same function, check it, and look at it.

In [ ]:
# The same function, a different category
tools = build_weekly("Tools")
tools_lines = sales[sales["Category"] == "Tools"]["Quantity"].sum()
print("Weeks:", len(tools), "| units in the weekly series:", int(tools["Units"].sum()), "| units in the sales file:", int(tools_lines))
print("Smallest week:", int(tools["Units"].min()), "| largest week:", int(tools["Units"].max()), "| mean:", round(tools["Units"].mean(), 1))

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(tools["date"], tools["Units"], color="#1A4C3B", linewidth=1.5)
ax.axvspan(tools.loc[TRAIN_WEEKS, "date"], tools["date"].iloc[-1], color="grey", alpha=0.15)
ax.set_ylabel("Units sold in the week")
ax.set_title("Tools: weekly units")
plt.show()

### Step 2: Run the model

Run the same comparison on Tools. The table has the same rows as before.

In [ ]:
tools_table = compare_methods(tools)
print(tools_table.round(1).to_string(index=False))

### Step 3: Read the output in plain English

Which drivers matter for Tools? Compare the coefficients and p-values for the two categories side by side, using the model that has all six drivers.

In [ ]:
# The same six-driver model on both categories: which drivers carry weight?
all_six = models["Regression: trend, season, promotions, school holidays, rain days, temperature"]
side_by_side = {}
for label, data in (("Barbecue", barbecue), ("Tools", tools)):
    fit = sm.OLS(data[data["t"] < TRAIN_WEEKS]["Units"], sm.add_constant(data[data["t"] < TRAIN_WEEKS][all_six])).fit()
    side_by_side[f"{label} coefficient"] = fit.params
    side_by_side[f"{label} p-value"] = fit.pvalues
print(pd.DataFrame(side_by_side).loc[["promo_pts", "SchoolHoliday", "RainDays", "AvgTempC"]].round(3).to_string())

In [ ]:
# The honest model for Tools: only the drivers you decided are known when the order is placed
tools_model, tools_forecast = fit_regression(tools, final_columns)
tools_score = score(tools.loc[tools["t"] >= TRAIN_WEEKS, "Units"], tools_forecast)
last_year_tools = tools_table.loc[tools_table["Method"] == "Same week last year", "MAPE %"].iloc[0]
print(f"Model in use: {', '.join(final_columns)}")
print(f"Tools: MAE {tools_score['MAE']:.1f} units, MAPE {tools_score['MAPE %']:.1f}%, against {last_year_tools:.1f}% for the same week last year.")

### Step 4: Check against JMP

Write the Tools table for JMP in the same format and fit the same model there.

In [ ]:
# Same layout as before: numeric Validation column (0 train, 1 test) and a Split label
tools[export_columns].to_csv("exports/tools_weekly_for_jmp.csv", index=False)
print("Written: exports/tools_weekly_for_jmp.csv", tools[export_columns].shape)
print(f"Python numbers to find in JMP: training R-squared {tools_model.rsquared:.3f}, test MAE {tools_score['MAE']:.1f} units")

### Step 5: Say it to a manager

Two sentences for a manager who runs both categories: how well can we forecast Tools, and does weather matter?

In [ ]:
# A draft built from the numbers. Your own words go in the next cell.
temp_p = side_by_side["Tools p-value"]["AvgTempC"]
print(f"Tools demand can be forecast to within about {tools_score['MAE']:.0f} units a week ({tools_score['MAPE %']:.1f}%), against {last_year_tools:.1f}% for repeating last year.")
print(f"Temperature has a p-value of {temp_p:.2f} for Tools, so unlike barbecues there is no evidence that weather moves it.")

*Your two sentences here*

### Commentary: Part 3

1. Which drivers matter for Tools, and which do not? Use the p-values.
2. Tools is easier to forecast than barbecues. Give one reason, using the drivers.
3. If you could buy a weather forecast for barbecues, what would you need to know about it before using it?

*Your commentary here*

### Your turn: point the code at another category

Change `"Tools"` to `"Plants and Seeds"` and run the Part 3 cells again. Does temperature matter there? Write down the p-value.

**Copilot:** write a prompt asking for *a loop that scores every category with the same function and prints one table*. Check each category's MAPE against the tables above before you trust the loop.

In [ ]:
# Copilot's loop goes here

---
## Stretch

A blank space for quick students. Ideas: ask Copilot for a stocking comparison on the Tools category; turn the whole Part 2 cost table into a function you can run on any category; or check how the intervals would change if you used the 90% point instead of the 0.75 point.

In [ ]:
# Stretch